In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

from warnings import filterwarnings
filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential

from tensorflow.keras.optimizers.legacy import Optimizer
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2

from tensorflow.keras.optimizers import AdamW
import pickle
from time import perf_counter


2025-04-21 04:44:07.961957: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-21 04:44:08.016166: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-21 04:44:08.017603: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-21 04:44:08.019630: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-21 04:44:08.028784: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-21 04:44:08.029750: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [2]:
counts = pd.read_csv('/work/counts_pred.csv')
X = pd.read_csv('/work/preprocessed.csv')
X['No Of Withdrawals'] = StandardScaler().fit_transform(counts)


y = pd.read_csv('/work/target.csv')['Total amount Withdrawn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

y_train_norm = (y_train - y_train.min()) / (y_train.max() - y_train.min())
y_test_norm = (y_test - y_test.min()) / (y_test.max() - y_test.min())


In [3]:
# Define the bootstrap cost function as a TensorFlow function

@keras.utils.register_keras_serializable()
def bootstrap_cost_function_tf(y_true, y_pred):

    residuals = tf.square(y_true - y_pred)  # (t_i - y_hat_i)^2
    variance_estimate = tf.math.reduce_variance(y_pred) #sigma_epsilon^2(x_i) - simplified for demonstration
    cost = 0.5 * tf.reduce_mean(tf.math.log(variance_estimate) + residuals / variance_estimate)
    return cost

In [191]:
def run_ann(X_train, y_train, epochs, validation_split):

    model = keras.Sequential([
        layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        layers.Dense(128, activation="relu"),
        Dropout(0.1),
        layers.Dense(32, activation="relu"),
        Dropout(0.1),
        BatchNormalization(),
        layers.Dense(1, activation="softplus"),
    ])

    optimizer = AdamW(learning_rate=1e-3)

    model.compile(optimizer=optimizer, loss=bootstrap_cost_function_tf, metrics=['mae'])
    history = model.fit(X_train, y_train, epochs=epochs, validation_split=validation_split, batch_size = 64, verbose=0)    

    return model, history


In [192]:
def train_bootstrap_models(data_X, data_y, n_models, epochs=200):

    models, histories = [], []
    for m in range(n_models):

        if m % 10 == 0:
            print('\nM:', m+1)
        
        # 1. Resample Data
        indices = np.random.choice(len(data_X), size=int(len(data_X)), replace=True)
        X_batch = data_X.values[indices, :]
        y_batch = data_y.values[indices]

        # 2. Train Neural Networks
        model, history = run_ann(X_batch, y_batch, epochs, validation_split=0.0)  # Use validation split()
        
        models.append(model)
        histories.append(history)

    return models, histories

In [193]:
def estimate_mean_prediction(models, data_X):
    predictions = np.array([model.predict(data_X, verbose=0) for model in models])
    mean_prediction = np.mean(predictions, axis=0)
    return mean_prediction


def estimate_model_variance(models, data_X, mean_prediction):
    predictions = np.array([model.predict(data_X, verbose=0) for model in models])
    model_variance = np.var(predictions, axis=0)
    return model_variance


def calculate_residuals(actual_y, predicted_y, model_variance):
   
    residuals = np.maximum((actual_y - predicted_y)**2 - model_variance, 0)
    return residuals


def estimate_noise_variance(variance_network, data_X):

    noise_variance = variance_network.predict(data_X, verbose=0)
    return noise_variance

In [420]:

def train_variance_network(X_test, residuals_squared):

    model = keras.Sequential([
        layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        layers.Dense(128, activation="relu",),
        Dropout(0.1),
        layers.Dense(32, activation="relu",),
        Dropout(0.1),
        BatchNormalization(),
        layers.Dense(1, activation="softplus"),
    ])

    optimizer = AdamW(learning_rate=1e-3)

    model.compile(optimizer=optimizer, loss=bootstrap_cost_function_tf, metrics=['mae'])
    history = model.fit(X_test, residuals_squared, epochs=200, batch_size = 32, verbose=0)    

    return model, history



In [203]:
def construct_prediction_interval(mean_prediction, model_variance, noise_variance, alpha=0.05):
    """
    Constructs the prediction interval.

    Args:
      mean_prediction: The average prediction.
      model_variance: The model misspecification variance.
      noise_variance: The estimated noise variance.
      alpha: The significance level (e.g., 0.05 for 95% CI).

    Returns:
      A tuple containing the lower and upper bounds of the prediction interval.
    """

    total_variance = model_variance + noise_variance
    std_dev = np.sqrt(total_variance)

    # Assuming t-distribution
    from scipy.stats import t
    df = len(mean_prediction) - 1  # Degrees of freedom (adjust as needed)
    t_value = t.ppf(1 - alpha / 2, df)

    lower_bound = mean_prediction - t_value * std_dev
    upper_bound = mean_prediction + t_value * std_dev

    return lower_bound, upper_bound

In [167]:
t1 = perf_counter()
# 2. Bootstrap and train models using the bootstrap cost function
bootstrap_models, histories = train_bootstrap_models(X_train, y_train_norm, n_models=30, epochs=100)



M: 1

M: 11

M: 21


In [168]:
t2 = perf_counter()
print(f'Time elsapsed: {t2-t1:.1f} seconds')

Time elsapsed: 718.7 seconds


In [206]:
# 3. Estimate mean and model variance
mean_prediction = estimate_mean_prediction(bootstrap_models, X_test)
mean_prediction = mean_prediction.flatten()
mean_prediction_denorm = mean_prediction * (y_test.max() - y_test.min()) + y_test.min()

In [423]:
model_variance = estimate_model_variance(bootstrap_models, X_test, mean_prediction_denorm)
model_variance = model_variance.flatten()
model_variance_denorm = model_variance * (y_test.max() - y_test.min()) + y_test.min()

In [424]:
 # 4. Train variance network (using MSE as in original code)


residuals_squared = calculate_residuals(y_test_norm, mean_prediction, model_variance)

variance_network, variance_history = train_variance_network(X_test, residuals_squared)


In [425]:
# 5. Estimate noise variance
noise_variance = estimate_noise_variance(variance_network, X_test)
noise_variance = noise_variance.flatten()
noise_variance_denorm = noise_variance * (y_test.max() - y_test.min()) + y_test.min()


In [426]:
 # 6. Construct prediction interval
lower_bound, upper_bound = construct_prediction_interval(mean_prediction, model_variance, noise_variance)


In [427]:
print("\nMean Prediction:", mean_prediction_denorm)

lb_amt = lower_bound * (y_test.max() - y_test.min()) + y_test.min()
ub_amt = upper_bound * (y_test.max() - y_test.min()) + y_test.min()

lb_amt = lb_amt.flatten().astype('int')
ub_amt = ub_amt.flatten().astype('int')

print("\nLower Bound:", lb_amt)
print("\nUpper Bound:", ub_amt)


Mean Prediction: [406551.62 266582.1  357837.28 ... 190593.02 451576.84 428671.1 ]

Lower Bound: [ 283882   51072  193943 ... -104519  330102  328069]

Upper Bound: [529221 482091 521730 ... 485705 573050 529272]


In [428]:
below_lb = (lb_amt >= y_test).sum() / len(y_test)
print('Points Below Lower Bound:', below_lb)

below_ub = (ub_amt >= y_test).sum() / len(y_test)
print('Points Below Upper Bound:', below_ub)


Points Below Lower Bound: 0.020118343195266272
Points Below Upper Bound: 0.9747534516765286


In [431]:
def compute_picp(y_lower, y_upper, y_true):
    return np.mean(np.logical_and(y_true >= y_lower, y_true <= y_upper))

def compute_pinrw(y_lower, y_upper, y_true):
    R = np.max(y_true) - np.min(y_true)
    width = np.sqrt(np.mean((y_upper - y_lower) ** 2))
    return width / R

def compute_pinrw(y_lower, y_upper, y_true):
    R = np.max(y_true) - np.min(y_true)
    width = np.sqrt(np.mean((y_upper - y_lower) ** 2))
    return width / R

def cwc_loss_numpy(y_lower, y_upper, y_true, gamma=50, eta=50, mu=0.95):
    picp = compute_picp(y_lower, y_upper, y_true)
    pinrw = compute_pinrw(y_lower, y_upper, y_true)
    penalty = np.exp(eta * (mu - picp))
    cwc = pinrw * (1 + gamma * penalty)
    mpiw = np.mean(y_upper - y_lower)

    return cwc, picp, mpiw, pinrw

In [434]:

cwc, picp, mpiw, pinrw = cwc_loss_numpy(lb_amt, ub_amt, y_test, gamma=50, eta=50, mu=0.95)
outcome = (y_test >= lb_amt) & (y_test <= ub_amt)

result = pd.DataFrame({'Original': y_test, 'Predicted': mean_prediction_denorm,
                        'Lower Bound': lb_amt, 'Upper Bound': ub_amt,
                        'Outcome': outcome})


result.to_csv(f'Results_Bootstrap_amt.csv', index=False)


In [437]:
print('PICP: ', picp)
print('PINRW: ', pinrw)
print('CWC: ', cwc)
print('MPIW: ', mpiw)

PICP:  0.9546351084812623
PINRW:  0.3211543784250849
CWC:  13.057174889443958
MPIW:  346893.8970414201


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=6c5dcd7e-ee44-459a-9dd6-5c39b2404cb3' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>